# Chapter 8: Multi-Provider LLM Comparison

**AI Enterprise Architecture — Companion Notebook**

This notebook sends the same prompts to **Gemini**, **Claude**, and **GPT-4** and compares:

- **Latency** — wall-clock time per request
- **Cost** — estimated cost per token from a pricing table
- **Consistency** — how similar two runs of the same prompt are

You will need API keys for all three providers. Set them as environment variables before running.

> **Key insight:** A multi-provider architecture provides resilience and cost optimization — the same pattern enterprise architects apply to multi-cloud infrastructure.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────
!pip install -q openai anthropic google-genai tabulate

### API Key Setup

This notebook reads API keys from environment variables. In Google Colab, you can set them via the Secrets panel (key icon in the left sidebar) or by running:

```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
os.environ["GOOGLE_API_KEY"] = "AI..."
```

**Never hard-code keys in a notebook you share.**

In [ ]:
import os
import time
import json
from difflib import SequenceMatcher

import openai
import anthropic
from google import genai
from tabulate import tabulate

# ── Validate keys are set ────────────────────────────────────────────
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY")
GOOGLE_KEY = os.environ.get("GOOGLE_API_KEY")

missing = []
if not OPENAI_KEY:    missing.append("OPENAI_API_KEY")
if not ANTHROPIC_KEY: missing.append("ANTHROPIC_API_KEY")
if not GOOGLE_KEY:    missing.append("GOOGLE_API_KEY")

if missing:
    print(f"WARNING: Missing environment variables: {', '.join(missing)}")
    print("Set them before running the provider calls.")
else:
    print("All API keys detected.")

---
## Define Test Prompts

Five enterprise-relevant prompts covering different task types.

In [ ]:
# ── Test Prompts ─────────────────────────────────────────────────────

PROMPTS = {
    "simple": "What is a service mesh? Answer in two sentences.",

    "summarization": (
        "Summarize the following paragraph in one sentence:\n\n"
        "Enterprise architecture frameworks such as TOGAF provide a structured "
        "approach to aligning IT strategy with business goals. They define "
        "principles, standards, and governance models that help large "
        "organisations manage complexity across hundreds of interconnected "
        "systems. However, adopting AI at scale introduces new dimensions — "
        "model lifecycle management, data provenance, and responsible AI — "
        "that traditional frameworks were not designed to address."
    ),

    "classification": (
        "Classify this support ticket into exactly one category "
        "(billing, technical, account, other):\n\n"
        "'I was charged twice for my March subscription and need a refund.'"
    ),

    "reasoning": (
        "A company runs its core ERP on-premises and wants to add an AI "
        "recommendation engine. The CTO insists on keeping all customer data "
        "in-region (EU). The ML team wants to use a US-based LLM API. "
        "What architecture would you recommend and why? Keep it under 100 words."
    ),

    "code": (
        "Write a Python function called `retry_with_backoff` that retries a "
        "callable up to `max_retries` times with exponential backoff. "
        "Include type hints and a docstring."
    ),
}

print(f"Defined {len(PROMPTS)} test prompts: {list(PROMPTS.keys())}")

---
## Provider Clients and Pricing

We define a thin wrapper for each provider so that every call returns the same shape:
`{"text", "input_tokens", "output_tokens", "latency_s"}`.

In [ ]:
# ── Pricing (USD per 1M tokens, as of early 2026) ────────────────────
# Update these when prices change.

PRICING = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "claude-sonnet": {"input": 3.00, "output": 15.00},
    "gemini-flash": {"input": 0.10, "output": 0.40},
}


def estimate_cost(provider: str, input_tokens: int, output_tokens: int) -> float:
    """Return estimated cost in USD for a single request."""
    p = PRICING[provider]
    return (input_tokens * p["input"] + output_tokens * p["output"]) / 1_000_000

In [ ]:
# ── Provider wrappers ────────────────────────────────────────────────

openai_client = openai.OpenAI(api_key=OPENAI_KEY) if OPENAI_KEY else None
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY) if ANTHROPIC_KEY else None
google_client = genai.Client(api_key=GOOGLE_KEY) if GOOGLE_KEY else None


def call_openai(prompt: str) -> dict:
    """Call GPT-4o and return standardised result."""
    start = time.time()
    resp = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
    )
    latency = time.time() - start
    return {
        "text": resp.choices[0].message.content,
        "input_tokens": resp.usage.prompt_tokens,
        "output_tokens": resp.usage.completion_tokens,
        "latency_s": round(latency, 2),
    }


def call_anthropic(prompt: str) -> dict:
    """Call Claude Sonnet and return standardised result."""
    start = time.time()
    resp = anthropic_client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
    )
    latency = time.time() - start
    return {
        "text": resp.content[0].text,
        "input_tokens": resp.usage.input_tokens,
        "output_tokens": resp.usage.output_tokens,
        "latency_s": round(latency, 2),
    }


def call_google(prompt: str) -> dict:
    """Call Gemini Flash and return standardised result."""
    start = time.time()
    resp = google_client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
    )
    latency = time.time() - start
    return {
        "text": resp.text,
        "input_tokens": resp.usage_metadata.prompt_token_count,
        "output_tokens": resp.usage_metadata.candidates_token_count,
        "latency_s": round(latency, 2),
    }


# Map of provider name -> (callable, pricing key)
PROVIDERS = {}
if openai_client:    PROVIDERS["gpt-4o"]        = call_openai
if anthropic_client: PROVIDERS["claude-sonnet"]  = call_anthropic
if google_client:    PROVIDERS["gemini-flash"]   = call_google

print(f"Active providers: {list(PROVIDERS.keys())}")

---
## Run the Comparison

Each prompt is sent to every provider **twice** so we can measure consistency between runs.

In [ ]:
# ── Run all prompts × all providers × 2 runs ────────────────────────

results = []  # list of dicts

for prompt_name, prompt_text in PROMPTS.items():
    for provider_name, call_fn in PROVIDERS.items():
        run_outputs = []
        for run_id in (1, 2):
            try:
                r = call_fn(prompt_text)
                cost = estimate_cost(provider_name, r["input_tokens"], r["output_tokens"])
                results.append({
                    "prompt": prompt_name,
                    "provider": provider_name,
                    "run": run_id,
                    "latency_s": r["latency_s"],
                    "input_tokens": r["input_tokens"],
                    "output_tokens": r["output_tokens"],
                    "cost_usd": round(cost, 6),
                    "text": r["text"],
                })
                run_outputs.append(r["text"])
                print(f"  OK  {prompt_name:15s} | {provider_name:15s} | run {run_id}")
            except Exception as e:
                print(f"  ERR {prompt_name:15s} | {provider_name:15s} | run {run_id} -> {e}")

print(f"\nCollected {len(results)} results.")

---
## Analysis: Comparison Tables

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

if df.empty:
    print("No results collected — check that at least one API key is set.")
else:
    # ── Table 1: Average metrics by provider ─────────────────────────
    by_provider = df.groupby("provider").agg(
        avg_latency_s=("latency_s", "mean"),
        avg_input_tokens=("input_tokens", "mean"),
        avg_output_tokens=("output_tokens", "mean"),
        avg_cost_usd=("cost_usd", "mean"),
    ).round(4).reset_index()

    print("=" * 60)
    print("Average Metrics by Provider")
    print("=" * 60)
    print(tabulate(by_provider, headers="keys", tablefmt="github", showindex=False))

In [ ]:
if not df.empty:
    # ── Table 2: Average metrics by prompt type ──────────────────────
    by_prompt = df.groupby(["prompt", "provider"]).agg(
        avg_latency_s=("latency_s", "mean"),
        avg_cost_usd=("cost_usd", "mean"),
    ).round(4).reset_index()

    print("\n" + "=" * 60)
    print("Average Metrics by Prompt Type and Provider")
    print("=" * 60)
    print(tabulate(by_prompt, headers="keys", tablefmt="github", showindex=False))

### Consistency Score

We compare Run 1 vs Run 2 for each (prompt, provider) pair using `SequenceMatcher` to compute a similarity ratio (1.0 = identical).

In [ ]:
if not df.empty:
    # ── Consistency: compare run 1 vs run 2 ─────────────────────────
    consistency_rows = []
    for (prompt_name, provider_name), group in df.groupby(["prompt", "provider"]):
        texts = group.sort_values("run")["text"].tolist()
        if len(texts) == 2:
            similarity = SequenceMatcher(None, texts[0], texts[1]).ratio()
            consistency_rows.append({
                "prompt": prompt_name,
                "provider": provider_name,
                "similarity": round(similarity, 3),
            })

    df_cons = pd.DataFrame(consistency_rows)
    print("\n" + "=" * 60)
    print("Consistency (Run 1 vs Run 2 similarity)")
    print("=" * 60)
    print(tabulate(df_cons, headers="keys", tablefmt="github", showindex=False))

    # Average consistency per provider
    avg_cons = df_cons.groupby("provider")["similarity"].mean().round(3)
    print("\nAverage consistency by provider:")
    for prov, score in avg_cons.items():
        print(f"  {prov}: {score}")

---
## Summary: Provider Recommendations

In [ ]:
if not df.empty:
    cheapest = by_provider.loc[by_provider["avg_cost_usd"].idxmin(), "provider"]
    fastest = by_provider.loc[by_provider["avg_latency_s"].idxmin(), "provider"]
    most_consistent = avg_cons.idxmax()

    print("=" * 60)
    print("PROVIDER SUMMARY")
    print("=" * 60)
    print(f"  Cheapest provider:         {cheapest}")
    print(f"  Fastest provider:          {fastest}")
    print(f"  Most consistent provider:  {most_consistent}")
    print()
    print("Recommendation: Use a routing layer that selects the optimal")
    print("provider per request based on latency budget, cost ceiling,")
    print("and task type. This is the AI equivalent of multi-CDN routing.")

---
## Takeaways

| Metric | Why It Matters |
|---|---|
| Latency | User-facing applications need sub-second responses. |
| Cost/token | At enterprise scale, small per-token differences compound fast. |
| Consistency | Deterministic outputs simplify testing and compliance. |

**Key insight:** Multi-provider architecture provides resilience and cost optimization — the same principle behind multi-cloud, applied to AI inference.